# Day 58 — PagedAttention as what it really is: a memory allocator for the KV cache

The attention change is small: read K and V through a block table instead of from one
flat array. What it enables is everything else — handing out fixed-size blocks, refusing
work when the pool is full, sharing blocks between samples of one prompt, and taking
blocks back from a running request. It is an operating system, not an attention variant.

In [1]:
import math

import numpy as np

# ================================================================ 1. a workload
def make_requests(n=240, seed=0, pool_tokens=8192):
    """Arrivals with a prompt and an answer length, the way a server sees them."""
    rng = np.random.default_rng(seed)
    reqs = []
    t = 0
    for i in range(n):
        t += int(rng.integers(0, 3))
        prompt = int(rng.integers(64, 1024))
        out = int(rng.integers(32, 512))
        reqs.append({'id': i, 'arrive': t, 'prompt': prompt, 'out': out,
                     'total': prompt + out})
    return reqs


# ================================================================ 2. two allocators
class ContiguousPool:
    """One contiguous range per request, reserved for the longest it might get.

    This is how you would write it if the KV cache were a single array per sequence:
    the range cannot move once handed out, so it has to be big enough from the start.
    """

    def __init__(self, capacity, reserve):
        self.capacity = capacity
        self.reserve = reserve          # tokens reserved per request
        self.free = [(0, capacity)]     # (start, length), kept sorted
        self.held = {}

    def allocate(self, rid):
        need = self.reserve
        best, bi = None, -1
        for i, (s, ln) in enumerate(self.free):      # best fit
            if ln >= need and (best is None or ln < best[1]):
                best, bi = (s, ln), i
        if best is None:
            return False
        s, ln = self.free.pop(bi)
        self.held[rid] = (s, need)
        if ln > need:
            self.free.insert(bi, (s + need, ln - need))
        return True

    def release(self, rid):
        s, ln = self.held.pop(rid)
        self.free.append((s, ln))
        self.free.sort()
        merged = []                                   # coalesce neighbours
        for seg in self.free:
            if merged and merged[-1][0] + merged[-1][1] == seg[0]:
                merged[-1] = (merged[-1][0], merged[-1][1] + seg[1])
            else:
                merged.append(seg)
        self.free = merged

    def free_tokens(self):
        return sum(ln for _, ln in self.free)

    def largest_hole(self):
        return max((ln for _, ln in self.free), default=0)


class PagedPool:
    """Fixed-size blocks and a block table per request - a page table, in other words."""

    def __init__(self, capacity, block=16):
        self.block = block
        self.n_blocks = capacity // block
        self.free = list(range(self.n_blocks))
        self.table = {}                  # rid -> [block ids]
        self.refs = {}                   # block id -> how many requests point at it

    def blocks_for(self, tokens):
        return math.ceil(tokens / self.block)

    def allocate(self, rid, tokens):
        need = self.blocks_for(tokens) - len(self.table.get(rid, []))
        if need <= 0:
            return True
        if need > len(self.free):
            return False
        got = [self.free.pop() for _ in range(need)]
        for b in got:
            self.refs[b] = 1
        self.table.setdefault(rid, []).extend(got)
        return True

    def share(self, src, dst, tokens):
        """Point a second request at the same blocks: parallel sampling, one prompt."""
        shared = self.table[src][:self.blocks_for(tokens)]
        for b in shared:
            self.refs[b] += 1
        self.table[dst] = list(shared)
        return len(shared)

    def release(self, rid):
        for b in self.table.pop(rid, []):
            self.refs[b] -= 1
            if self.refs[b] == 0:
                self.free.append(b)

    def used_blocks(self):
        return self.n_blocks - len(self.free)

    def free_tokens(self):
        return len(self.free) * self.block


# ================================================================ 3. what paging buys
def fragmentation(reqs, capacity=16384, block=16):
    """Serve the same arrivals both ways and count who gets turned away.

    The contiguous pool here is given the best possible case: it is told the exact
    final length of every request, so it never over-reserves.  It still refuses work,
    because the free space it has is in the wrong shape - that is external
    fragmentation, and it is the thing fixed-size blocks remove by construction.
    """
    cont = ContiguousPool(capacity, 0)
    paged = PagedPool(capacity, block)
    running_c, running_p = {}, {}
    rej_c = rej_p = rej_c_frag = 0
    use_c, use_p = [], []
    clock = 0
    pending = list(reqs)
    while pending or running_c or running_p:
        clock += 1
        for rid, (end, _) in list(running_c.items()):
            if end <= clock:
                cont.release(rid)
                del running_c[rid]
        for rid, (end, _) in list(running_p.items()):
            if end <= clock:
                paged.release(rid)
                del running_p[rid]
        while pending and pending[0]['arrive'] <= clock:
            r = pending.pop(0)
            life = clock + r['total'] // 64 + 1
            cont.reserve = r['total']                 # exact size: the kindest case
            if cont.allocate(r['id']):
                running_c[r['id']] = (life, r['total'])
            else:
                rej_c += 1
                if cont.free_tokens() >= r['total']:  # room, but not in one piece
                    rej_c_frag += 1
            if paged.allocate(r['id'], r['total']):
                running_p[r['id']] = (life, r['total'])
            else:
                rej_p += 1
        use_c.append(sum(t for _, t in running_c.values()))
        use_p.append(sum(t for _, t in running_p.values()))
    return {'rejected_contiguous': rej_c, 'rejected_fragmentation': rej_c_frag,
            'rejected_paged': rej_p, 'util_contiguous': float(np.mean(use_c)) / capacity,
            'util_paged': float(np.mean(use_p)) / capacity, 'steps': clock,
            'requests': len(reqs)}


reqs = make_requests()
f = fragmentation(reqs)
print(f"{f['requests']} requests, a 16,384-slot pool")
print(f"contiguous (told the exact length!): {f['rejected_contiguous']} refused, "
      f"{f['rejected_fragmentation']} of them with enough free memory")
print(f"paged, 16 tokens per block        : {f['rejected_paged']} refused")
print(f"occupancy over time: {f['util_contiguous'] * 100:.1f}% vs {f['util_paged'] * 100:.1f}%")

240 requests, a 16,384-slot pool
contiguous (told the exact length!): 12 refused, 12 of them with enough free memory
paged, 16 tokens per block        : 5 refused
occupancy over time: 65.7% vs 68.5%


## What paging costs instead: the tail of the last block

In [2]:
def internal_waste(reqs, blocks=(1, 8, 16, 64, 256)):
    """What paging costs: the unused tail of every request's last block."""
    rows = []
    for b in blocks:
        used = sum(math.ceil(r['total'] / b) * b for r in reqs)
        real = sum(r['total'] for r in reqs)
        rows.append({'block': b, 'slots': used, 'waste': (used - real) / used,
                     'tables': sum(math.ceil(r['total'] / b) for r in reqs)})
    return rows


for r in internal_waste(reqs):
    print(f"block {r['block']:>4}: {r['waste'] * 100:4.1f}% wasted, "
          f"{r['tables']:>7,} block-table entries")

block    1:  0.0% wasted, 201,736 block-table entries
block    8:  0.4% wasted,  25,318 block-table entries
block   16:  0.9% wasted,  12,721 block-table entries
block   64:  3.5% wasted,   3,266 block-table entries
block  256: 12.5% wasted,     901 block-table entries


## Sharing one prompt between samples, and copy-on-write

The samples point at the same blocks, kept alive by a reference count. Only the last
block of the prompt is a problem: it is partly full, so each sample that writes its own
next token into it needs its own copy.

In [3]:
# ================================================================ 4. sharing
def parallel_sampling(prompt=900, n_samples=4, out=200, block=16, capacity=65536):
    """n answers to one prompt: the prompt's blocks are shared, not copied.

    The last prompt block is the interesting one: it is only partly full, so the moment
    two samples write their own next token into it, it has to be copied.  That is
    copy-on-write, and it is one block per sample, not one prompt per sample.
    """
    pool = PagedPool(capacity, block)
    pool.allocate('s0', prompt)
    shared = 0
    for i in range(1, n_samples):
        shared += pool.share('s0', f's{i}', prompt)
    naive = n_samples * math.ceil(prompt / block)
    cow = 1 if prompt % block else 0           # the partial block each sample copies
    actual = pool.used_blocks() + cow * (n_samples - 1)
    for i in range(n_samples):
        pool.allocate(f's{i}', prompt + out)
    return {'prompt_blocks': math.ceil(prompt / block), 'naive': naive,
            'shared_blocks': shared, 'with_sharing': actual,
            'saved': 1 - actual / naive, 'cow_blocks': cow * (n_samples - 1),
            'after_decode': pool.used_blocks()}


ps = parallel_sampling()
print(f"one copy per sample : {ps['naive']} blocks")
print(f"sharing the prompt  : {ps['with_sharing']} blocks ({ps['saved'] * 100:.0f}% less)")
print(f"copied on write     : {ps['cow_blocks']} blocks")

one copy per sample : 228 blocks
sharing the prompt  : 60 blocks (74% less)
copied on write     : 3 blocks


## Preemption: compute or bandwidth

When the pool runs out, a running request has to give its blocks back — either dropped
and re-prefilled later, or copied to host memory and copied back.

In [4]:
# ================================================================ 5. preemption
def preemption(capacity=4096, block=16, n=60, seed=1, policy='recompute',
               max_steps=20000):
    """When the pool runs out, a running request has to give its blocks back.

    Two ways to do it: drop them and recompute the prefix later, or copy them to host
    memory and copy them back.  The first costs compute, the second costs bandwidth -
    the same trade as the day before, one level up.

    Newest-first is the victim rule, because the newest request has the least work
    invested in it; it is also what vLLM does.
    """
    rng = np.random.default_rng(seed)
    pool = PagedPool(capacity, block)
    waiting = [{'id': i, 'need': int(rng.integers(200, 900)), 'done': 0}
               for i in range(n)]
    running = []
    recomputed = swapped = steps = preemptions = 0
    while (waiting or running) and steps < max_steps:
        steps += 1
        while waiting and pool.allocate(waiting[0]['id'], waiting[0]['need']):
            running.append(waiting.pop(0))
        for r in list(running):                       # one decode step of work
            r['done'] += 64
            if r['done'] >= r['need']:
                pool.release(r['id'])
                running.remove(r)
        if waiting and running and pool.free_tokens() < waiting[0]['need']:
            victim = running.pop()                    # newest first
            pool.release(victim['id'])
            preemptions += 1
            if policy == 'recompute':
                recomputed += victim['done']          # work thrown away
                victim['done'] = 0
            else:
                swapped += victim['done']             # copied out and back
            waiting.append(victim)
    return {'preemptions': preemptions, 'recomputed_tokens': recomputed,
            'swapped_tokens': swapped, 'steps': steps,
            'finished': n - len(waiting) - len(running)}


a, b = preemption(policy='recompute'), preemption(policy='swap')
print(f"recompute: {a['preemptions']} preemptions, {a['recomputed_tokens']:,} tokens re-prefilled")
print(f"swap     : {b['preemptions']} preemptions, {b['swapped_tokens']:,} tokens moved")
print(f"both finished all {a['finished']} requests")

recompute: 54 preemptions, 3,840 tokens re-prefilled
swap     : 51 preemptions, 5,696 tokens moved
both finished all 60 requests


## Two silent failures

In [5]:
# ================================================================ 6. two silent failures
def leaked_blocks(n=200, block=16, capacity=65536, fail_rate=0.1, seed=0):
    """A request that errors out and never frees its blocks.

    Nothing raises: the pool just gets smaller, and the server slowly stops accepting
    work for no visible reason.
    """
    rng = np.random.default_rng(seed)
    pool = PagedPool(capacity, block)
    leaked = 0
    for i in range(n):
        tokens = int(rng.integers(200, 800))
        pool.allocate(i, tokens)
        if rng.random() < fail_rate:
            leaked += pool.blocks_for(tokens)      # the except: branch forgot to free
        else:
            pool.release(i)
    return {'leaked_blocks': leaked, 'leaked_tokens': leaked * block,
            'free_left': len(pool.free) * block, 'capacity': capacity}


def block_table_off_by_one(tokens=70, block=16):
    """The last block is partly full, and the kernel has to be told how full.

    Round the length up to the block and attention reads the slots after the last real
    token - uninitialised memory, not a crash.
    """
    n_blocks = math.ceil(tokens / block)
    padded = n_blocks * block
    return {'tokens': tokens, 'blocks': n_blocks, 'padded': padded,
            'garbage_slots': padded - tokens,
            'share': (padded - tokens) / padded}


lk = leaked_blocks()
print(f"leaked {lk['leaked_blocks']} blocks = {lk['leaked_tokens']:,} slots of "
      f"{lk['capacity']:,}; pool left {lk['free_left']:,}")
ob = block_table_off_by_one()
print(f"{ob['tokens']} tokens -> {ob['blocks']} blocks = {ob['padded']} slots, "
      f"{ob['garbage_slots']} of them uninitialised")

leaked 617 blocks = 9,872 slots of 65,536; pool left 55,664
70 tokens -> 5 blocks = 80 slots, 10 of them uninitialised


## The assertions

In [6]:
# ================================================================ 7. assertions
def check():
    reqs = make_requests()

    # paging turns away far fewer requests than a contiguous pool of the same size
    f = fragmentation(reqs)
    assert f['rejected_paged'] < f['rejected_contiguous']
    assert f['rejected_fragmentation'] == f['rejected_contiguous'] > 0
    assert f['util_paged'] > f['util_contiguous']

    # internal waste falls as the block gets smaller, and the table gets longer
    rows = internal_waste(reqs)
    assert rows[0]['waste'] == 0 and rows[-1]['waste'] > rows[1]['waste']
    assert rows[0]['tables'] > rows[-1]['tables']

    # sharing one prompt between samples saves most of the blocks
    ps = parallel_sampling()
    assert ps['with_sharing'] < ps['naive'] and ps['saved'] > 0.5

    # both preemption policies move the same work, in different currencies
    a = preemption(policy='recompute')
    b = preemption(policy='swap')
    assert a['preemptions'] > 0 and b['preemptions'] > 0
    assert a['recomputed_tokens'] > 0 and b['swapped_tokens'] > 0
    assert a['finished'] == b['finished'] == 60

    # a leak is silent: the pool shrinks and nothing raises
    lk = leaked_blocks()
    assert lk['leaked_tokens'] > 0 and lk['free_left'] < lk['capacity']

    # the last block is partly garbage unless the kernel is told the real length
    ob = block_table_off_by_one()
    assert ob['garbage_slots'] > 0
    return True


print('all assertions passed' if check() else 'FAILED')

all assertions passed
